# Libraries and env

In [1]:
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

In [10]:
from dotenv import load_dotenv
import os
from pathlib import Path

def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  # Loads .env from project root (works if run from notebook too)
    env_vars = {
        "IMAGE_FOLDER": os.getenv("IMAGE_FOLDER"),
        "EXTRACTEDDATASET_FOLDER": os.getenv("EXTRACTEDDATASET_FOLDER"),
        "DATASETS_FOLDER": os.getenv("DATASETS_FOLDER"),
        "ElevationDataset": os.getenv("ElevationDataset"),
        "LandCoverDataset": os.getenv("LandCoverDataset"),
        "GeoBoundaries": os.getenv("GeoBoundaries"),
        "EXTRACTEDELEVATION_FOLDER": os.getenv("EXTRACTEDELEVATION_FOLDER"),
        "EXTRACTEDLANDCOVER_FOLDER": os.getenv("EXTRACTEDLANDCOVER_FOLDER"),
        "EXTRACTEDGEOBOUNDARIES_FOLDER": os.getenv("EXTRACTEDGEOBOUNDARIES_FOLDER"),
        "CLEANEDDATASET_FOLDER": os.getenv("CLEANEDDATASET_FOLDER"),
        "CLEANEDELEVATION_FOLDER": os.getenv("CLEANEDELEVATION_FOLDER"),
        "CLEANEDLANDCOVER_FOLDER": os.getenv("CLEANEDLANDCOVER_FOLDER"),
        "PREPROCESSED_DATASET_FOLDER": os.getenv("PREPROCESSED_DATASET_FOLDER"),
        "PREPROCESSED_ELEVATION_FOLDER": os.getenv("PREPROCESSED_ELEVATION_FOLDER"),
        "PREPROCESSED_LANDCOVER_FOLDER": os.getenv("PREPROCESSED_LANDCOVER_FOLDER")
    }
    return env_vars

folders = load_environment()
image_folder = folders["IMAGE_FOLDER"]
extractedData_folder = folders["EXTRACTEDDATASET_FOLDER"]
datasets_folder = folders["DATASETS_FOLDER"]
landCover_folder = folders["LandCoverDataset"]
elevation_folder = folders["ElevationDataset"]
geoboundaries_folder = folders["GeoBoundaries"]
extracted_elevation_folder = folders["EXTRACTEDELEVATION_FOLDER"]
extracted_landcover_folder = folders["EXTRACTEDLANDCOVER_FOLDER"]
extracted_geo_boundaries_folder = folders["EXTRACTEDGEOBOUNDARIES_FOLDER"]
cleaned_dataset_folder = folders["CLEANEDDATASET_FOLDER"]
cleaned_elevation_folder = folders["CLEANEDELEVATION_FOLDER"]
cleaned_landcover_folder = folders["CLEANEDLANDCOVER_FOLDER"]
preprocessed_dataset_folder = folders["PREPROCESSED_DATASET_FOLDER"]
preprocessed_elevation_folder = folders["PREPROCESSED_ELEVATION_FOLDER"]
preprocessed_landcover_folder = folders["PREPROCESSED_LANDCOVER_FOLDER"]


# Read file

In [3]:
# retrieve dataset from cleaned folder
land_cover = gpd.read_file(os.path.join(cleaned_landcover_folder, "landcover_cleaned.geojson"))
land_cover.head()


c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Several features with id = 1 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


,id,area,lcc_highlevel,geometry
0,4,6.228187e+06,Water bodies,"POLYGON ((6.41528 37.08696, 6.43103 37.0855, 6..."
1,2,6.242408e+06,Water bodies,"POLYGON ((7.18084 37.07917, 7.17998 37.08091, ..."
2,1,1.482995e+06,Water bodies,"POLYGON ((7.37137 37.08194, 7.3709 37.08717, 7..."
3,8,4.590841e+08,Forests,"POLYGON ((6.12361 36.68472, 6.12361 36.69306, ..."
4,13,6.371533e+06,Water bodies,"POLYGON ((6.26181 37.02361, 6.26193 37.02514, ..."


In [4]:
#  type of each column
land_cover.dtypes


id                  int32
area              float64
lcc_highlevel      object
geometry         geometry
dtype: object

In [5]:
#  lets check if area needs scaling
land_cover["area"].describe()


count    4.385130e+05
mean     5.636630e+06
std      1.192254e+09
min      1.002300e+05
25%      1.727640e+05
50%      3.102027e+05
75%      6.763858e+05
max      6.720004e+11
Name: area, dtype: float64

a) Range is enormous

Your smallest value ≈ 10⁵, while your largest ≈ 10¹¹ — a difference of six orders of magnitude.

b) Outliers dominate

That huge max means the data is highly skewed.
Even after scaling, you might want to handle outliers separately (log-transform or winsorization).

# Preprocessing

## Scaling area column -- recheck if should be left after splitting

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
land_cover['area_scaled'] = scaler.fit_transform(land_cover[['area']])
land_cover[['area', 'area_scaled']].head()

,area,area_scaled
0,6.228187e+06,0.000496
1,6.242408e+06,0.000508
2,1.482995e+06,-0.003484
3,4.590841e+08,0.380328
4,6.371533e+06,0.000616


## Hot encoding lcc high level

In [7]:
#  hot encoding lcc_highlevel into a vector of binary columns
land_cover_encoded = pd.get_dummies(
    land_cover, 
    columns=['lcc_highlevel'], 
    prefix='lcc', 
    drop_first=True
)

land_cover_encoded.head()




,id,area,geometry,area_scaled,lcc_Bare lands,lcc_Croplands,lcc_Forests,lcc_Grasslands,lcc_Vegetation,lcc_Water bodies
0,4,6.228187e+06,"POLYGON ((6.41528 37.08696, 6.43103 37.0855, 6...",0.000496,False,False,False,False,False,True
1,2,6.242408e+06,"POLYGON ((7.18084 37.07917, 7.17998 37.08091, ...",0.000508,False,False,False,False,False,True
2,1,1.482995e+06,"POLYGON ((7.37137 37.08194, 7.3709 37.08717, 7...",-0.003484,False,False,False,False,False,True
3,8,4.590841e+08,"POLYGON ((6.12361 36.68472, 6.12361 36.69306, ...",0.380328,False,False,True,False,False,False
4,13,6.371533e+06,"POLYGON ((6.26181 37.02361, 6.26193 37.02514, ...",0.000616,False,False,False,False,False,True


## missing data

In [8]:
# checking missing data
land_cover_encoded.isnull().sum()


id                  0
area                0
geometry            0
area_scaled         0
lcc_Bare lands      0
lcc_Croplands       0
lcc_Forests         0
lcc_Grasslands      0
lcc_Vegetation      0
lcc_Water bodies    0
dtype: int64

## polygon column

In [ ]:
#  drop area column
land_cover_final = land_cover_encoded.drop(columns='area')

In [14]:
land_cover_final.head()

,id,area_scaled,lcc_Bare lands,lcc_Croplands,lcc_Forests,lcc_Grasslands,lcc_Vegetation,lcc_Water bodies,perimeter,centroid_x,centroid_y
0,4,0.000496,False,False,False,False,False,True,0.339045,6.484240,37.076069
1,2,0.000508,False,False,False,False,False,True,0.315034,7.234732,37.078333
2,1,-0.003484,False,False,False,False,False,True,0.072981,7.383729,37.084692
3,8,0.380328,False,False,True,False,False,False,9.595515,6.339666,36.907812
4,13,0.000616,False,False,False,False,False,True,0.389813,6.335560,37.059089


In [15]:
# saving land_cover_final to preprocessed_landcover_folder
land_cover_final.to_csv(os.path.join(preprocessed_landcover_folder, "preprocessed_landcover.csv"), index=False)
